<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/decision_tree-train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Exited"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Exited"]
)

# Cluster treinado SÓ com cancelados do df_treino (nunca olha teste/validação)
df_cancelados_treino = df_treino[df_treino["Exited"] == 1][variaveis_numericas]

scaler_perfil = StandardScaler()
X_scaled_cancelados = scaler_perfil.fit_transform(df_cancelados_treino)

kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_k4.fit(X_scaled_cancelados)

# Aplicado (predict) nas 3 bases separadamente, sem re-treinar em nenhuma delas
for dataframe in [df_treino, df_teste, df_validacao]:
    X_scaled = scaler_perfil.transform(dataframe[variaveis_numericas])
    dataframe['Cluster_k4'] = kmeans_k4.predict(X_scaled)
    dataframe['Distancia_Centroide_k4'] = kmeans_k4.transform(X_scaled).min(axis=1)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

features = variaveis_numericas + ['Cluster_k4', 'Distancia_Centroide_k4']

X_train = df_treino[features]
y_train = df_treino['Exited']
X_test = df_teste[features]
y_test = df_teste['Exited']

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1
)
grid_search.fit(X_train, y_train)

melhor_modelo = grid_search.best_estimator_
y_pred = melhor_modelo.predict(X_test)
y_proba = melhor_modelo.predict_proba(X_test)[:, 1]

resultado_arvore_k4 = {
    'melhores_parametros': grid_search.best_params_,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
}
pd.DataFrame([resultado_arvore_k4])